# Capítulo 8 · Delineamento de estudos observacionais em saúde: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, uma população de 24.000 operações em 12 hospitais que adotaram um pacote de prevenção de infecção em momentos diferentes, pelas perguntas da diretoria de qualidade; aparecem só as seções com cálculo (8.3, 8.5 a 8.9, 8.11 e 8.12). O livro desenha as figuras; o caderno imprime os mesmos números. A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 12; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (3, 4, 5, 7 e 10), o código e o resultado. O código é Python 3 com pandas. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/population.csv`: 24.000 linhas, uma por operação, com o hospital, o mês, o diabetes e o tabagismo do paciente (verdadeiro e relatado), a classe da cirurgia, o pacote (verdadeiro e como registrado), a infecção em 30 dias (verdadeira e como classificada por dois avaliadores) e se o seguimento foi completo. `data/hospitals.csv`: os 12 hospitais, com a proporção de operações contaminadas e o mês de adoção do pacote (vazio quando nunca adotado). `data/glucose_pairs.csv`: leituras pareadas de glicose de um glicosímetro e de um laboratório. Cada célula traz o número da seção do livro a que pertence.

### 8.3 A população que construímos

*Quantas operações, quantas infecções e quando cada hospital adotou o pacote?*

In [1]:
# 1. Ler a população de 24.000 operações em 12 hospitais (campos em branco do registro ficam como texto vazio, não como valores faltantes)
import pandas as pd

df = pd.read_csv("data/population.csv", keep_default_na=False)
print("Operações:", len(df), "; proporção seguida de infecção em 30 dias:", round((df["ssi_30d"] == "yes").mean(), 4))

# 2. Os 12 hospitais: proporção de operações contaminadas e mês de adoção do pacote (vazio quando nunca foi adotado)
hosp = pd.read_csv("data/hospitals.csv", keep_default_na=False)
print(hosp.to_string(index=False))

Operações: 24000 ; proporção seguida de infecção em 30 dias: 0.1011
hospital  contaminated_share bundle_adoption_month
     H01                0.25                      
     H02                0.30                      
     H03                0.35                      
     H04                0.40                    21
     H05                0.45                    19
     H06                0.50                    17
     H07                0.55                    15
     H08                0.60                    13
     H09                0.65                    13
     H10                0.70                    11
     H11                0.75                     9
     H12                0.80                     7


### 8.5 O estudo ecológico

*Os hospitais que adotaram o pacote têm mais ou menos infecções?*

In [2]:
# 1. Uma linha por hospital: a proporção de operações contaminadas, de pacientes com o pacote e de infecções (em %)
flags = pd.DataFrame({"hospital": df["hospital"],
                      "contaminated": df["surgery_class"] == "contaminated",
                      "bundle": df["bundle"] == "yes",
                      "infection": df["ssi_30d"] == "yes"})
by_hospital = flags.groupby("hospital").mean()
print((100 * by_hospital).round(1).head(3))

# 2. A correlação entre hospitais da proporção com o pacote e da proporção de infecções
print("Correlação entre a proporção com o pacote e a proporção de infecções:", round(by_hospital["bundle"].corr(by_hospital["infection"]), 2))

          contaminated  bundle  infection
hospital                                 
H01               26.2     0.0        8.2
H02               31.0     0.0        9.8
H03               33.2     0.0        7.8
Correlação entre a proporção com o pacote e a proporção de infecções: 0.72


### 8.6 O estudo de coorte

*Entre os pacientes acompanhados desde a adoção do pacote, ele se associa a menos infecções?*

In [3]:
# 1. A coorte: as operações a partir do mês em que cada hospital adotou o pacote (99 para os três que nunca adotaram)
adopt = {}
for row in hosp.itertuples():
    if row.bundle_adoption_month:
        adopt[row.hospital] = int(row.bundle_adoption_month)
    else:
        adopt[row.hospital] = 99
cohort = df[df["month"] >= df["hospital"].map(adopt)]

# 2. A tabela dois por dois do pacote e da infecção na coorte e a razão de riscos do pacote
t = pd.crosstab(cohort["bundle"], cohort["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
risk_ratio = (a / (a + b)) / (c / (c + d))
print("Operações na coorte:", len(cohort), "; células a, b, c, d:", int(a), int(b), int(c), int(d), "; razão de riscos:", round(risk_ratio, 2))

# 3. A razão de riscos do pacote dentro de cada combinação de classe de cirurgia e diabetes
for cls in ["clean", "contaminated"]:
    for dm in ["yes", "no"]:
        stratum = cohort[(cohort["surgery_class"] == cls) & (cohort["diabetes"] == dm)]
        t = pd.crosstab(stratum["bundle"], stratum["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
        a, b = t.loc["yes"]
        c, d = t.loc["no"]
        print("Classe", cls, "; diabetes", dm, ": razão de riscos", round((a / (a + b)) / (c / (c + d)), 2))

Operações na coorte: 8240 ; células a, b, c, d: 444 5463 232 2101 ; razão de riscos: 0.76
Classe clean ; diabetes yes : razão de riscos 0.36
Classe clean ; diabetes no : razão de riscos 0.45
Classe contaminated ; diabetes yes : razão de riscos 0.49
Classe contaminated ; diabetes no : razão de riscos 0.44


In [4]:
# 4. Só os pacientes com seguimento completo: as células e a razão de riscos
complete = cohort[cohort["followup_complete"] == "yes"]
t = pd.crosstab(complete["bundle"], complete["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Registros completos:", len(complete), "; células a, b, c, d:", int(a), int(b), int(c), int(d), "; razão de riscos:", round((a / (a + b)) / (c / (c + d)), 2))

# 5. A proporção de pacientes sem e com seguimento completo, por pacote e infecção
lost = pd.crosstab([cohort["bundle"], cohort["ssi_30d"]], cohort["followup_complete"], normalize="index")
print(lost.round(3))

Registros completos: 7736 ; células a, b, c, d: 406 5185 152 1993 ; razão de riscos: 1.02
followup_complete     no    yes
bundle ssi_30d                 
no     no          0.051  0.949
       yes         0.345  0.655
yes    no          0.051  0.949
       yes         0.086  0.914


### 8.7 O estudo de caso-controle

*O que um estudo de caso-controle dos mesmos dados fornece, e e se os controles vierem da fonte errada?*

In [5]:
# 1. Todos os casos de infecção na coorte e dois controles por caso sorteados entre os pacientes da coorte sem infecção (semente 8)
cases = cohort[cohort["ssi_30d"] == "yes"]
controls = cohort[cohort["ssi_30d"] == "no"].sample(n=2 * len(cases), random_state=8)
sample = pd.concat([cases, controls])
t = pd.crosstab(sample["bundle"], sample["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Casos:", len(cases), "; células a, b, c, d:", int(a), int(b), int(c), int(d), "; razão de chances:", round(a * d / (b * c), 2))

# 2. A comparação inválida: o mesmo número de controles sorteados da população inteira, incluindo hospitais que nunca adotaram o pacote
wrong = df[df["ssi_30d"] == "no"].sample(n=2 * len(cases), random_state=8)
sample = pd.concat([cases, wrong])
t = pd.crosstab(sample["bundle"], sample["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Controles inválidos - células a, b, c, d:", int(a), int(b), int(c), int(d), "; razão de chances:", round(a * d / (b * c), 2))

Casos: 676 ; células a, b, c, d: 444 969 232 383 ; razão de chances: 0.76
Controles inválidos - células a, b, c, d: 444 337 232 1015 ; razão de chances: 5.76


### 8.8 Dados do mundo real e registros

*O que acontece quando os brancos de um registro são lidos como ausência de tratamento?*

In [6]:
# 1. Recodificar todo branco do campo de pacote do registro como "no", contando antes os brancos (a decisão silenciosa tornada visível)
registry = cohort.copy()
registry["bundle_registry"] = registry["bundle_recorded"].replace("", "no")
print("Campos em branco:", (cohort["bundle_recorded"] == "").sum(), "; proporção de brancos entre os pacientes que receberam o pacote:", round((cohort.loc[cohort["bundle"] == "yes", "bundle_recorded"] == "").mean(), 3))

# 2. A razão de riscos do pacote com o campo do registro
t = pd.crosstab(registry["bundle_registry"], registry["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Razão de riscos com o campo do registro:", round((a / (a + b)) / (c / (c + d)), 2))

Campos em branco: 745 ; proporção de brancos entre os pacientes que receberam o pacote: 0.126
Razão de riscos com o campo do registro: 0.81


### 8.9 As três famílias de erro sistemático

*Como seleção, informação e confusão mudam a estimativa?*

In [7]:
# 1. A razão de riscos do tabagismo para infecção na população inteira, com o tabagismo verdadeiro e com o relatado
for column in ["smoker", "smoker_reported"]:
    t = pd.crosstab(df[column], df["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
    a, b = t.loc["yes"]
    c, d = t.loc["no"]
    print("Razão de riscos do tabagismo, usando", column + ":", round((a / (a + b)) / (c / (c + d)), 2))

# 2. Tabagismo verdadeiro contra relatado: a tabela de classificação errada que um investigador não vê
print(pd.crosstab(df["smoker"], df["smoker_reported"]))

# 3. A razão de riscos do pacote na coorte com a infecção classificada pelo avaliador A
assessed = cohort.assign(ssi_30d=cohort["ssi_assessor_a"])
t = pd.crosstab(assessed["bundle"], assessed["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
a, b = t.loc["yes"]
c, d = t.loc["no"]
print("Razão de riscos do pacote com o avaliador A:", round((a / (a + b)) / (c / (c + d)), 2))

Razão de riscos do tabagismo, usando smoker: 2.06
Razão de riscos do tabagismo, usando smoker_reported: 1.81
smoker_reported     no   yes
smoker                      
no               16232   357
yes               1557  5854
Razão de riscos do pacote com o avaliador A: 0.8


### 8.11 Concordância entre avaliadores: concordância percentual e kappa

*Quão bem dois avaliadores concordam, e quão válido é cada um?*

In [8]:
# 1. A tabela dos dois avaliadores e a concordância observada
t = pd.crosstab(df["ssi_assessor_a"], df["ssi_assessor_b"]).loc[["yes", "no"], ["yes", "no"]]
print(t)
n = len(df)
p_o = (t.loc["yes", "yes"] + t.loc["no", "no"]) / n

# 2. A concordância esperada ao acaso a partir das proporções marginais de cada avaliador e o kappa
row = t.sum(axis=1) / n
col = t.sum(axis=0) / n
p_e = row["yes"] * col["yes"] + row["no"] * col["no"]
print("Concordância observada:", round(p_o, 3), "; esperada ao acaso:", round(p_e, 3), "; kappa:", round((p_o - p_e) / (1 - p_e), 2))

# 3. A validade do avaliador A contra a situação de infecção conhecida: sensibilidade e especificidade
v = pd.crosstab(df["ssi_assessor_a"], df["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
print("Sensibilidade do avaliador A:", round(v.loc["yes", "yes"] / v["yes"].sum(), 3), "; especificidade:", round(v.loc["no", "no"] / v["no"].sum(), 3))

ssi_assessor_b   yes     no
ssi_assessor_a             
yes             1721   1009
no               703  20567
Concordância observada: 0.929 ; esperada ao acaso: 0.808 ; kappa: 0.63
Sensibilidade do avaliador A: 0.863 ; especificidade: 0.971


### 8.12 Concordância entre métodos: análise de Bland-Altman

*O glicosímetro e o laboratório concordam?*

In [9]:
# 1. Leituras pareadas de glicose: o glicosímetro e o laboratório
g = pd.read_csv("data/glucose_pairs.csv")
diff = g["glucose_meter"] - g["glucose_lab"]
print("Correlação entre o laboratório e o glicosímetro:", round(g["glucose_lab"].corr(g["glucose_meter"]), 2))

# 2. Bland-Altman: a diferença média e os limites de concordância (média mais ou menos 1,96 desvio padrão)
print("Diferença média:", round(diff.mean(), 1), "; desvio padrão:", round(diff.std(), 1), "; limites:", round(diff.mean() - 1.96 * diff.std(), 1), "a", round(diff.mean() + 1.96 * diff.std(), 1))

Correlação entre o laboratório e o glicosímetro: 0.96
Diferença média: 6.6 ; desvio padrão: 8.7 ; limites: -10.5 a 23.6


## Parte B. Soluções dos exercícios

### 8.16 Exercício 1. Mobilização precoce e pneumonia

População: adultos submetidos a cirurgia de quadril no hospital. Exposição:
mobilização em até 24 horas após a cirurgia, em comparação com mobilização mais tardia.
Desfecho: pneumonia segundo uma definição declarada. Tempo: trinta dias a partir da cirurgia.
O experimento-alvo incluiria os pacientes elegíveis, atribuiria mobilização precoce ou tardia
por sorteio e acompanharia todos por trinta dias. Um estudo a partir dos registros da enfermaria
se afasta dele em pelo menos dois pontos: a mobilização é decidida pela
equipe, de modo que os pacientes mais frágeis são mobilizados tarde (confusão por
indicação); e o relógio começa na cirurgia para todos, mas um paciente que
desenvolve pneumonia no dia 1 pode ter a mobilização precoce adiada, de modo que a classificação pelo tempo eventual de mobilização pode depender de um desfecho
anterior. As estratégias-alvo e o tratamento da janela de 24 horas
devem ser especificados antes do seguimento; simplesmente reclassificar os pacientes depois
não restaura a comparação randomizada.

### 8.16 Exercício 2. O que uma fotografia pode medir

A proporção de pacientes internados com ferida infectada: sim, é uma prevalência
entre os presentes. O risco em trinta dias: não, exige o seguimento de todas as
operações, e a fotografia sobrerrepresenta as internações longas. A proporção de
operações do mês com o pacote: somente se o pacote for registrado para
todos os operados naquele mês, o que é uma questão de registro, e não de levantamento;
os pacientes internados presentes não são as operações do mês. Se o
pacote reduz o risco: não, pelas duas razões acima e porque o retrato
não consegue ordenar exposição e desfecho no tempo.

### 8.16 Exercício 3. Diabetes entre hospitais e dentro dos pacientes

A correlação no nível dos hospitais é 0,35. O diabetes foi gerado com a
mesma probabilidade de 25% em todos os hospitais, de modo que suas proporções por hospital
variam apenas pela amostra realizada, enquanto a infecção também varia com
o perfil de casos e outras características do hospital; a razão de risco no nível
individual, 2,52, compara diretamente pacientes diabéticos com não diabéticos. Uma
responde “hospitais com mais diabéticos têm mais infecções” e a
outra “pacientes diabéticos se infectam com mais frequência”.

In [10]:
# 1. Entre hospitais: a correlação entre a proporção com diabetes e a proporção de infecções
flags = pd.DataFrame({"hospital": df["hospital"], "diabetes": df["diabetes"] == "yes", "infection": df["ssi_30d"] == "yes"})
by_h = flags.groupby("hospital").mean()
print("Correlação entre hospitais:", round(by_h["diabetes"].corr(by_h["infection"]), 2))

# 2. Entre pacientes: a razão de riscos do diabetes para infecção
t = pd.crosstab(df["diabetes"], df["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
print("Razão de riscos do diabetes:", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Correlação entre hospitais: 0.35
Razão de riscos do diabetes: 2.52


### 8.16 Exercício 4. Estratificando por hospital

Nove hospitais adotaram o pacote; suas razões vão de 0,54 a 1,18 e
apenas uma fica abaixo de 0,60. Estratificar mantém fixa a pertença ao hospital;
isso não torna receptores e não receptores comparáveis quanto ao risco do paciente
ou ao tempo calendário dentro de cada hospital. As razões também variam
por acaso. Dentro desta coorte elegível, a estratificação por classe de cirurgia e diabetes
controla a confusão codificada no gerador. Suas estimativas ainda
variam por acaso e não precisam ser exatamente iguais ao efeito gerador.

In [11]:
# 1. A razão de riscos do pacote dentro de cada hospital da coorte
for h in sorted(cohort["hospital"].unique()):
    sub = cohort[cohort["hospital"] == h]
    t = pd.crosstab(sub["bundle"], sub["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
    print("Hospital", h + ": razão de riscos", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Hospital H04: razão de riscos 1.18
Hospital H05: razão de riscos 0.99
Hospital H06: razão de riscos 0.79
Hospital H07: razão de riscos 0.83
Hospital H08: razão de riscos 0.61
Hospital H09: razão de riscos 0.85
Hospital H10: razão de riscos 0.79
Hospital H11: razão de riscos 0.54
Hospital H12: razão de riscos 0.74


### 8.16 Exercício 5. Rastreando metade das perdas

Rastrear essa metade aleatória move a razão de 1,02 para 0,89, em direção à
associação da coorte completa de 0,76. Não há regra segundo a qual rastrear metade
remova metade do viés: uma razão de risco combina numeradores e
denominadores que mudam de forma não linear, e a amostra rastreada varia por acaso.
O grupo não rastreado ainda pode ser sistematicamente diferente. Rastrear apenas
pacientes fáceis de contatar não precisa ter o mesmo resultado.

In [12]:
# 1. Rastrear metade dos pacientes perdidos no seguimento (semente 8) e juntá-los aos registros completos
lost = cohort[cohort["followup_complete"] == "no"]
traced = lost.sample(frac=0.5, random_state=8)
known = pd.concat([cohort[cohort["followup_complete"] == "yes"], traced])

# 2. A razão de riscos do pacote entre os registros conhecidos
t = pd.crosstab(known["bundle"], known["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
print("Registros conhecidos:", len(known), "; razão de riscos:", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Registros conhecidos: 7988 ; razão de riscos: 0.89


### 8.16 Exercício 6. Controles de um ambulatório

A população de origem dos casos são os pacientes operados nos doze
hospitais durante o período do estudo, já que qualquer um deles poderia ter se tornado
caso. Os pacientes ambulatoriais de uma clínica não são essa população: não foram
operados e vêm de uma única área de abrangência. Se a exposição fosse mais comum
nessa área, os controles seriam mais frequentemente expostos do que a população de
origem, a razão de chances seria empurrada para perto de 1 ou abaixo dele para uma exposição
nociva, e o erro seria viés de seleção, não reparado simplesmente ajustando para as covariáveis medidas. São necessárias melhor amostragem
dos controles ou informação adicional sobre o mecanismo de seleção.

### 8.16 Exercício 7. O campo em branco

Ler os brancos como “não” dá 0,81 e como “sim” dá 0,76. O segundo
acerta, aqui, porque todo branco pertence a um receptor verdadeiro,
mas o analista não pode saber isso a partir dos dados; o dicionário de dados deve dizer
o que um branco significa (não administrado ou não registrado) e, se não puder, a
análise relata ambos.

In [13]:
# 1. Recodificar o campo em branco do registro como "no" e depois como "yes", e comparar as razões de riscos do pacote
registry = cohort.copy()
for value in ["no", "yes"]:
    registry["b"] = registry["bundle_recorded"].replace("", value)
    t = pd.crosstab(registry["b"], registry["ssi_30d"]).loc[["yes", "no"], ["yes", "no"]]
    print("Brancos recodificados como", value + ": razão de riscos", round((t.loc["yes", "yes"] / t.loc["yes"].sum()) / (t.loc["no", "yes"] / t.loc["no"].sum()), 2))

Brancos recodificados como no: razão de riscos 0.81
Brancos recodificados como yes: razão de riscos 0.76


### 8.16 Exercício 8. Três famílias

(a) Viés de seleção: voluntários diferem dos não voluntários de maneiras relacionadas ao
desfecho. (b) Viés de informação, diferencial (viés de memória): a
exposição é medida de forma mais completa nos casos. (c) Confusão por
indicação: a gravidade está associada à prescrição e afeta a
mortalidade. (d) Viés de seleção: perda relacionada tanto à exposição quanto ao desfecho.
(e) Viés de informação diferencial: a pressão arterial é o desfecho, e seu
erro de medida depende da obesidade, a exposição.

### 8.16 Exercício 9. Duas enfermeiras e 300 lesões

Porcentagem de concordância $p_o=(42+228)/300=0{,}900$. A enfermeira A classifica 60/300 = 0,200
como presente e a enfermeira B 54/300 = 0,180, de modo que
$p_e=0{,}200\times0{,}180+0{,}800\times0{,}820=0{,}692$ e
$\kappa=(0{,}900-0{,}692)/(1-0{,}692)=0{,}68$. A mudança da prevalência de lesões pode alterar tanto a concordância observada quanto a
concordância esperada pelas margens, mesmo que a sensibilidade e a
especificidade das enfermeiras permaneçam constantes. A tabela não identifica a prevalência verdadeira
nem as taxas de erro de nenhuma das enfermeiras. Portanto, ela não pode determinar o tamanho nem a
direção de uma mudança no kappa em uma segunda enfermaria.

### 8.16 Exercício 10. Quarenta miligramas somados a cada leitura do glicosímetro

A correlação permanece inalterada em 0,96; a diferença média passa de 6,6 para
46,6 mg/dL e os limites de concordância de $-10{,}5$ a $23{,}6$ para $29{,}5$ a
$63{,}6$; o desvio padrão das diferenças permanece inalterado. O glicosímetro modificado lê 46,6 mg/dL a mais em média, com variação
substancial entre pacientes, e ainda assim mantém a mesma correlação alta: a correlação mede se os dois sobem juntos, e não
se concordam.

In [14]:
# 1. Quarenta miligramas somados a cada leitura do glicosímetro: a correlação, a diferença média e os limites de concordância
meter = g["glucose_meter"] + 40
diff = meter - g["glucose_lab"]
print("Correlação:", round(g["glucose_lab"].corr(meter), 2), "; diferença média:", round(diff.mean(), 1), "; desvio padrão:", round(diff.std(), 1), "; limites:", round(diff.mean() - 1.96 * diff.std(), 1), "a", round(diff.mean() + 1.96 * diff.std(), 1))

Correlação: 0.96 ; diferença média: 46.6 ; desvio padrão: 8.7 ; limites: 29.5 a 63.6


### 8.16 Exercício 11. Sensibilidade e especificidade do autorrelato

A partir da tabela, 5.854 dos 7.411 fumantes verdadeiros relatam fumar
(sensibilidade 0,79) e 16.232 dos 16.589 não fumantes relatam não
fumar (especificidade 0,98). O grupo de fumantes relatados contém, portanto, alguns
não fumantes e, mais importante, o grupo de não fumantes relatados contém
1.557 fumantes verdadeiros com o risco de um fumante. Cada grupo é uma mistura dos
dois grupos verdadeiros, e uma mistura de dois riscos fica entre eles, de modo que a
razão observada, 1,81, fica entre a associação dos dados completos de 2,06 e 1 nesta simulação. Esta não é uma
regra universal de direção do viés para todos os contextos de erro de medida.

### 8.16 Exercício 12. Escolhendo o delineamento

(a) Levantamento transversal com medida de glicose: prevalência de
diabetes não diagnosticado. (b) Estudo de caso-controle, já que a reação é rara:
razão de chances para a variante. (c) Coorte de trabalhadores do turno noturno e
do turno diurno, idealmente a partir de registros ocupacionais com longo seguimento:
taxas e razões de taxas para cada câncer. (d) Um experimento entre os
serviços, o delineamento randomizado por conglomerados do Capítulo 10;
na falta dele, um estudo controlado de antes e depois: diferença no tempo médio de
espera. (e) Um estudo de concordância com medidas pareadas nos mesmos
pacientes: diferença média e limites de concordância, e não uma correlação.

---
*Dados sintéticos (`generate_data.py`, semente 8). Valores apenas didáticos; nada aqui é estimativa para uma rede hospitalar real.*